<a href="https://colab.research.google.com/github/karanwq/Heart_disease_predictor/blob/main/health_with_ai_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y torch torchvision torchaudio torchcodec
!pip uninstall -y transformers accelerate sentencepiece sentence-transformers faiss-cpu

Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: accelerate 0.31.0
Uninstalling accelerate-0.31.0:
  Successfully uninstalled accelerate-0.31.0
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
Found existing installation: faiss-cpu 1.13.2
Uninstalling faiss-cpu-1.13.2:
  Successfully uninstalled faiss-cpu-1.13.2


In [2]:
!pip install torch==2.2.2
!pip install transformers==4.41.2 accelerate==0.31.0 sentencepiece
!pip install sentence-transformers==2.7.0 faiss-cpu flask pyngrok

  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl (755.5 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.26 requires torchvision, which is not installed.
peft 0.18.1 requires accelerate>=0.21.0, which is not installed.
peft 0.18.1 requires transformers, which is not installed.
torchtune 0.6.1 requires sentencepiece, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-0.31.0-py3-none-any.whl.metadata (19 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
Using cached accelerate-0.31.0-py3-none-any.whl (309 kB)
Us

In [3]:
import torch, transformers, sentence_transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

Torch: 2.2.2+cu121
Transformers: 4.41.2


In [4]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")

print(generator("What causes heart disease?", max_new_tokens=50))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[{'generated_text': 'a faulty heart'}]


In [5]:
import numpy as np
import pandas as pd
import pickle
import os

from flask import Flask, render_template, request
from pyngrok import ngrok

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [6]:
from google.colab import files
uploaded = files.upload()

Saving heart.csv to heart (5).csv


In [7]:
df = pd.read_csv("heart.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train_scaled, y_train)

print("Model trained")

Model trained


In [8]:
with open('health.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model & Scaler saved")

Model & Scaler saved


In [17]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

documents = [
    "High blood pressure can occur due to stress, high salt intake, lack of exercise, or obesity.",
    "Cholesterol buildup in arteries reduces blood flow and increases heart disease risk.",
    "Smoking damages blood vessels and increases blood pressure.",
    "Regular exercise helps lower blood pressure and improves heart health.",
    "Obesity increases strain on the heart and raises blood pressure.",
    "Excess salt intake causes water retention which increases blood pressure.",
    "Diabetes damages blood vessels and contributes to heart disease."
]

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = embed_model.encode(documents)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

def retrieve(query, k=5):
    q_emb = embed_model.encode([query])
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)

    D, I = index.search(q_emb, k)
    return [documents[i] for i in I[0]]

In [10]:
import pickle

with open('health.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model & Scaler saved ")

Model & Scaler saved 


In [11]:
os.makedirs("templates", exist_ok=True)

In [12]:
import shap

In [13]:
html_code = """
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Heart AI Predictor</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">

<style>
  :root {
    --red-50: #FCEBEB;
    --red-100: #F7C1C1;
    --red-200: #F09595;
    --red-400: #E24B4A;
    --red-600: #A32D2D;
    --red-800: #791F1F;
    --teal-50: #E1F5EE;
    --teal-400: #1D9E75;
    --teal-600: #0F6E56;
    --amber-400: #BA7517;
    --gray-50: #F7F6F2;
    --gray-100: #ECEAE3;
    --gray-200: #D6D4CC;
    --gray-400: #888780;
    --gray-800: #2E2D2B;
  }

  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    font-family: 'DM Sans', sans-serif;
    background: #F4F2EC;
    color: #2E2D2B;
    min-height: 100vh;
    overflow-x: hidden;
  }

  /* Light background texture */
  body::before {
    content: '';
    position: fixed;
    inset: 0;
    background:
      radial-gradient(ellipse 80% 60% at 20% 10%, rgba(226, 75, 74, 0.07) 0%, transparent 60%),
      radial-gradient(ellipse 60% 50% at 80% 80%, rgba(29, 158, 117, 0.06) 0%, transparent 55%),
      radial-gradient(ellipse 40% 40% at 60% 30%, rgba(186, 117, 23, 0.04) 0%, transparent 50%);
    pointer-events: none;
    z-index: 0;
  }

  .page-wrap {
    position: relative;
    z-index: 1;
    min-height: 100vh;
    display: flex;
    flex-direction: column;
    align-items: center;
    padding: 3rem 1.5rem 5rem;
  }

  /* ── Header ── */
  .header {
    text-align: center;
    margin-bottom: 3rem;
    animation: fadeDown 0.7s ease both;
  }

  .header-eyebrow {
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 0.25em;
    text-transform: uppercase;
    color: var(--red-400);
    margin-bottom: 1rem;
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 10px;
  }

  .header-eyebrow::before,
  .header-eyebrow::after {
    content: '';
    display: block;
    width: 40px;
    height: 1px;
    background: var(--red-400);
    opacity: 0.5;
  }

  .header h1 {
    font-family: 'DM Serif Display', serif;
    font-size: clamp(2.4rem, 5vw, 3.6rem);
    font-weight: 400;
    line-height: 1.1;
    color: #2E2D2B;
    margin-bottom: 0.75rem;
  }

  .header h1 em {
    font-style: italic;
    color: var(--red-400);
  }

  .header-sub {
    font-size: 15px;
    color: rgba(46, 45, 43, 0.5);
    font-weight: 300;
    letter-spacing: 0.01em;
    max-width: 380px;
    margin: 0 auto;
    line-height: 1.6;
  }

  /* ── Card ── */
  .card {
    width: 100%;
    max-width: 820px;
    background: #FFFFFF;
    border: 1px solid rgba(46, 45, 43, 0.08);
    border-radius: 24px;
    overflow: hidden;
    box-shadow: 0 4px 24px rgba(46, 45, 43, 0.07);
    animation: fadeUp 0.8s 0.15s ease both;
  }

  /* ── Section labels ── */
  .section-label {
    font-size: 10px;
    font-weight: 600;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    color: rgba(46, 45, 43, 0.4);
    padding: 1.5rem 2rem 0.75rem;
    border-bottom: 1px solid rgba(46, 45, 43, 0.06);
    display: flex;
    align-items: center;
    gap: 8px;
  }

  .section-label .dot {
    width: 6px;
    height: 6px;
    border-radius: 50%;
    background: var(--red-400);
    opacity: 0.7;
  }

  /* ── Form grid ── */
  .form-body {
    padding: 1.5rem 2rem 2rem;
  }

  .fields-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));
    gap: 12px;
    margin-bottom: 1.75rem;
  }

  .field-wrap {
    display: flex;
    flex-direction: column;
    gap: 6px;
  }

  .field-label {
    font-size: 11px;
    font-weight: 500;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: rgba(46, 45, 43, 0.5);
  }

  .field-hint {
    font-size: 10px;
    color: rgba(46, 45, 43, 0.35);
    font-weight: 300;
  }

  input[type="text"],
  input[type="number"],
  input {
    width: 100%;
    padding: 10px 14px;
    border-radius: 10px;
    border: 1px solid rgba(46, 45, 43, 0.14);
    background: #F7F6F2;
    color: #2E2D2B;
    font-family: 'DM Sans', sans-serif;
    font-size: 14px;
    font-weight: 400;
    outline: none;
    transition: border-color 0.2s, background 0.2s;
    -webkit-appearance: none;
  }

  input::placeholder {
    color: rgba(46, 45, 43, 0.3);
  }

  input:focus {
    border-color: rgba(226, 75, 74, 0.5);
    background: #fff;
  }

  input:hover:not(:focus) {
    border-color: rgba(46, 45, 43, 0.25);
  }

  /* ── Submit button ── */
  .btn-predict {
    width: 100%;
    padding: 15px 28px;
    border-radius: 12px;
    border: none;
    background: var(--red-400);
    color: #fff;
    font-family: 'DM Sans', sans-serif;
    font-size: 14px;
    font-weight: 600;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    cursor: pointer;
    position: relative;
    overflow: hidden;
    transition: transform 0.15s, opacity 0.2s;
  }

  .btn-predict::after {
    content: '';
    position: absolute;
    inset: 0;
    background: linear-gradient(to right, rgba(255,255,255,0.1), transparent);
    pointer-events: none;
  }

  .btn-predict:hover { opacity: 0.88; transform: translateY(-1px); }
  .btn-predict:active { transform: scale(0.99); }

  /* ── Divider ── */
  .divider {
    height: 1px;
    background: rgba(46, 45, 43, 0.07);
    margin: 0 2rem;
  }

  /* ── Result area ── */
  .result-section {
    padding: 1.75rem 2rem 2rem;
  }

  .result-headline {
    font-family: 'DM Serif Display', serif;
    font-size: 1.6rem;
    font-weight: 400;
    color: #2E2D2B;
    margin-bottom: 1.25rem;
    line-height: 1.3;
  }

  .risk-meter-wrap {
    margin-bottom: 1.5rem;
  }

  .risk-meta {
    display: flex;
    justify-content: space-between;
    align-items: baseline;
    margin-bottom: 10px;
  }

  .risk-meta-label {
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    color: rgba(46, 45, 43, 0.4);
    font-weight: 500;
  }

  .risk-percentage {
    font-family: 'DM Serif Display', serif;
    font-size: 2rem;
    color: #2E2D2B;
    line-height: 1;
  }

  .risk-percentage span {
    font-family: 'DM Sans', sans-serif;
    font-size: 13px;
    color: rgba(46, 45, 43, 0.4);
    font-weight: 300;
    margin-left: 3px;
  }

  .risk-track {
    height: 8px;
    background: rgba(46, 45, 43, 0.08);
    border-radius: 100px;
    overflow: hidden;
    position: relative;
  }

  .risk-fill {
    height: 100%;
    width: 0%;
    border-radius: 100px;
    background: linear-gradient(90deg, var(--teal-400) 0%, var(--amber-400) 55%, var(--red-400) 100%);
    transition: width 1.2s cubic-bezier(0.4, 0, 0.2, 1);
    position: relative;
  }

  .risk-fill::after {
    content: '';
    position: absolute;
    right: 0;
    top: 50%;
    transform: translateY(-50%);
    width: 14px;
    height: 14px;
    border-radius: 50%;
    background: #fff;
    box-shadow: 0 0 0 3px rgba(0,0,0,0.1);
  }

  .risk-scale {
    display: flex;
    justify-content: space-between;
    margin-top: 6px;
  }

  .risk-scale-item {
    font-size: 10px;
    color: rgba(46, 45, 43, 0.35);
    letter-spacing: 0.04em;
  }

  /* Status chips */
  .status-chips {
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
  }

  .chip {
    padding: 5px 14px;
    border-radius: 100px;
    font-size: 12px;
    font-weight: 500;
    letter-spacing: 0.04em;
  }

  .chip-low {
    background: rgba(29, 158, 117, 0.1);
    color: var(--teal-600);
    border: 1px solid rgba(29, 158, 117, 0.25);
  }

  .chip-medium {
    background: rgba(186, 117, 23, 0.1);
    color: #8A5A0F;
    border: 1px solid rgba(186, 117, 23, 0.25);
  }

  .chip-high {
    background: rgba(226, 75, 74, 0.1);
    color: var(--red-600);
    border: 1px solid rgba(226, 75, 74, 0.25);
  }

  .footer-note {
    margin-top: 2.5rem;
    text-align: center;
    font-size: 12px;
    color: rgba(46, 45, 43, 0.35);
    font-weight: 300;
    line-height: 1.7;
    max-width: 500px;
  }

  .footer-note strong {
    color: rgba(46, 45, 43, 0.5);
    font-weight: 500;
  }

  /* ── Animations ── */
  @keyframes fadeDown {
    from { opacity: 0; transform: translateY(-18px); }
    to   { opacity: 1; transform: translateY(0); }
  }

  @keyframes fadeUp {
    from { opacity: 0; transform: translateY(22px); }
    to   { opacity: 1; transform: translateY(0); }
  }

  @media (max-width: 580px) {
    .form-body { padding: 1.25rem 1.25rem 1.5rem; }
    .section-label { padding: 1.25rem 1.25rem 0.75rem; }
    .divider { margin: 0 1.25rem; }
    .result-section { padding: 1.5rem 1.25rem; }
    .fields-grid { grid-template-columns: 1fr 1fr; gap: 10px; }
  }

  @media (max-width: 400px) {
    .fields-grid { grid-template-columns: 1fr; }
  }

  /* Pulse heart decoration */
  .heart-icon {
    display: inline-block;
    width: 32px; height: 32px;
    margin-bottom: 1.25rem;
    animation: heartbeat 1.6s ease-in-out infinite;
  }

  @keyframes heartbeat {
    0%, 100% { transform: scale(1); }
    14% { transform: scale(1.2); }
    28% { transform: scale(1); }
    42% { transform: scale(1.12); }
    70% { transform: scale(1); }
  }

  /* ECG line decoration */
  .ecg-line {
    width: 100%;
    max-width: 340px;
    margin: 0.75rem auto 0;
    opacity: 0.25;
  }

  /* ── Chatbot Widget ── */
  #chat-toggle {
    position: fixed;
    bottom: 28px;
    right: 28px;
    z-index: 1000;
    width: 56px;
    height: 56px;
    border-radius: 50%;
    border: none;
    background: var(--red-400);
    color: #fff;
    cursor: pointer;
    box-shadow: 0 4px 18px rgba(226,75,74,0.35);
    display: flex;
    align-items: center;
    justify-content: center;
    transition: transform 0.2s, box-shadow 0.2s;
  }
  #chat-toggle:hover { transform: scale(1.08); box-shadow: 0 6px 24px rgba(226,75,74,0.45); }

  #chat-window {
    position: fixed;
    bottom: 96px;
    right: 28px;
    z-index: 999;
    width: 340px;
    max-height: 500px;
    background: #fff;
    border: 1px solid rgba(46,45,43,0.1);
    border-radius: 20px;
    box-shadow: 0 8px 40px rgba(46,45,43,0.13);
    display: none;
    flex-direction: column;
    overflow: hidden;
    animation: fadeUp 0.25s ease both;
  }
  #chat-window.open { display: flex; }

  .chat-header {
    background: var(--red-400);
    padding: 14px 18px;
    display: flex;
    align-items: center;
    gap: 10px;
  }
  .chat-header-icon {
    width: 30px; height: 30px;
    background: rgba(255,255,255,0.2);
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0;
  }
  .chat-header-text h3 {
    font-size: 13px;
    font-weight: 600;
    color: #fff;
    line-height: 1.2;
  }
  .chat-header-text p {
    font-size: 11px;
    color: rgba(255,255,255,0.7);
    font-weight: 300;
  }

  #chat-messages {
    flex: 1;
    overflow-y: auto;
    padding: 14px 14px 8px;
    display: flex;
    flex-direction: column;
    gap: 10px;
    background: #F7F6F2;
  }

  .msg {
    max-width: 85%;
    padding: 9px 13px;
    border-radius: 14px;
    font-size: 13px;
    line-height: 1.5;
  }
  .msg.bot {
    background: #fff;
    color: #2E2D2B;
    border: 1px solid rgba(46,45,43,0.1);
    border-bottom-left-radius: 4px;
    align-self: flex-start;
  }
  .msg.user {
    background: var(--red-400);
    color: #fff;
    border-bottom-right-radius: 4px;
    align-self: flex-end;
  }
  .msg.typing {
    background: #fff;
    border: 1px solid rgba(46,45,43,0.1);
    color: rgba(46,45,43,0.4);
    align-self: flex-start;
    font-style: italic;
    font-size: 12px;
  }

  .chat-input-row {
    display: flex;
    gap: 8px;
    padding: 10px 12px;
    background: #fff;
    border-top: 1px solid rgba(46,45,43,0.07);
  }
  #chat-input {
    flex: 1;
    border: 1px solid rgba(46,45,43,0.14);
    border-radius: 10px;
    padding: 9px 12px;
    font-size: 13px;
    font-family: 'DM Sans', sans-serif;
    color: #2E2D2B;
    background: #F7F6F2;
    outline: none;
    resize: none;
  }
  #chat-input:focus { border-color: rgba(226,75,74,0.45); background: #fff; }
  #chat-send {
    width: 36px; height: 36px;
    border-radius: 10px;
    border: none;
    background: var(--red-400);
    color: #fff;
    cursor: pointer;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0;
    transition: opacity 0.2s;
    align-self: flex-end;
  }
  #chat-send:hover { opacity: 0.85; }
</style>
</head>
<body>

<div class="page-wrap">

  <!-- Header -->
  <header class="header">
    <svg class="heart-icon" viewBox="0 0 32 32" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path d="M16 28C16 28 3 20.5 3 11.5C3 7.36 6.13 4 10 4C12.35 4 14.43 5.21 16 7C17.57 5.21 19.65 4 22 4C25.87 4 29 7.36 29 11.5C29 20.5 16 28 16 28Z" fill="#E24B4A"/>
    </svg>
    <div class="header-eyebrow">AI Clinical Assessment</div>
    <h1>Heart Risk<br><em>Predictor</em></h1>
    <p class="header-sub">Enter patient clinical data below for an AI-powered cardiovascular risk assessment.</p>
    <svg class="ecg-line" viewBox="0 0 340 28" xmlns="http://www.w3.org/2000/svg">
      <polyline points="0,14 40,14 52,14 60,2 68,26 74,6 82,22 90,14 130,14 170,14 210,14 250,14 290,14 340,14" fill="none" stroke="#E24B4A" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </svg>
  </header>

  <!-- Main card -->
  <div class="card">

    <div class="section-label">
      <span class="dot"></span>
      Patient Clinical Parameters
    </div>

    <form action="/predict" method="POST">
      <div class="form-body">

        <div class="fields-grid">

          <div class="field-wrap">
            <label class="field-label" for="age">Age <span class="field-hint">years</span></label>
            <input type="number" id="age" name="age" placeholder="e.g. 54" min="1" max="120">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="sex">Sex <span class="field-hint">0=F · 1=M</span></label>
            <input type="number" id="sex" name="sex" placeholder="0 or 1" min="0" max="1">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="cp">Chest Pain <span class="field-hint">type 0–3</span></label>
            <input type="number" id="cp" name="cp" placeholder="0–3" min="0" max="3">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="trestbps">Resting BP <span class="field-hint">mm Hg</span></label>
            <input type="number" id="trestbps" name="trestbps" placeholder="e.g. 120">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="chol">Cholesterol <span class="field-hint">mg/dl</span></label>
            <input type="number" id="chol" name="chol" placeholder="e.g. 220">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="fbs">Fasting Blood Sugar <span class="field-hint">&gt;120 mg/dl</span></label>
            <input type="number" id="fbs" name="fbs" placeholder="0 or 1" min="0" max="1">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="restecg">Rest ECG <span class="field-hint">0–2</span></label>
            <input type="number" id="restecg" name="restecg" placeholder="0–2" min="0" max="2">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="thalach">Max Heart Rate <span class="field-hint">bpm</span></label>
            <input type="number" id="thalach" name="thalach" placeholder="e.g. 150">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="exang">Exercise Angina <span class="field-hint">0=No · 1=Yes</span></label>
            <input type="number" id="exang" name="exang" placeholder="0 or 1" min="0" max="1">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="oldpeak">ST Depression <span class="field-hint">Oldpeak</span></label>
            <input type="number" id="oldpeak" name="oldpeak" placeholder="e.g. 1.2" step="0.1">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="slope">Slope <span class="field-hint">0–2</span></label>
            <input type="number" id="slope" name="slope" placeholder="0–2" min="0" max="2">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="ca">Major Vessels <span class="field-hint">0–4</span></label>
            <input type="number" id="ca" name="ca" placeholder="0–4" min="0" max="4">
          </div>

          <div class="field-wrap">
            <label class="field-label" for="thal">Thalassemia <span class="field-hint">1–3</span></label>
            <input type="number" id="thal" name="thal" placeholder="1–3" min="1" max="3">
          </div>

        </div>

        <button type="submit" class="btn-predict">Run Prediction &rarr;</button>

      </div>
    </form>

    {% if prediction_text %}

    <div class="divider"></div>

    <div class="section-label">
      <span class="dot"></span>
      Assessment Result
    </div>

    <div class="result-section">

      <p class="result-headline">{{ prediction_text }}</p>

      <div class="risk-meter-wrap">
        <div class="risk-meta">
          <span class="risk-meta-label">Cardiovascular Risk Score</span>
          <div class="risk-percentage">{{ risk }}<span>%</span></div>
        </div>

        <div class="risk-track">
          <div class="risk-fill" id="riskFill"></div>
        </div>

        <div class="risk-scale">
          <span class="risk-scale-item">Low</span>
          <span class="risk-scale-item">Moderate</span>
          <span class="risk-scale-item">High</span>
        </div>
      </div>

      <div class="status-chips">
        {% if risk|int < 30 %}
        <span class="chip chip-low">Low Risk</span>
        <span class="chip chip-low">Routine Monitoring</span>
        {% elif risk|int < 65 %}
        <span class="chip chip-medium">Moderate Risk</span>
        <span class="chip chip-medium">Follow-Up Recommended</span>
        {% else %}
        <span class="chip chip-high">High Risk</span>
        <span class="chip chip-high">Urgent Review Advised</span>
        {% endif %}
      </div>

    </div>

    {% endif %}

  </div>

  <p class="footer-note">
    <strong>Clinical use disclaimer.</strong> This tool provides a statistical estimate only and is not a substitute for professional medical diagnosis. Always consult a qualified cardiologist.
  </p>

</div>

<button id="chat-toggle" title="Ask a health question" aria-label="Open chat">
  <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
    <path d="M21 15C21 15.5304 20.7893 16.0391 20.4142 16.4142C20.0391 16.7893 19.5304 17 19 17H7L3 21V5C3 4.46957 3.21071 3.96086 3.58579 3.58579C3.96086 3.21071 4.46957 3 5 3H19C19.5304 3 20.0391 3.21071 20.4142 3.58579C20.7893 3.96086 21 4.46957 21 5V15Z" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
  </svg>
</button>

<div id="chat-window">
  <div class="chat-header">
    <div class="chat-header-icon">
      <svg width="16" height="16" viewBox="0 0 32 32" fill="none">
        <path d="M16 28C16 28 3 20.5 3 11.5C3 7.36 6.13 4 10 4C12.35 4 14.43 5.21 16 7C17.57 5.21 19.65 4 22 4C25.87 4 29 7.36 29 11.5C29 20.5 16 28 16 28Z" fill="white"/>
      </svg>
    </div>
    <div class="chat-header-text">
      <h3>Heart Health Assistant</h3>
      <p>Powered by RAG · Ask anything</p>
    </div>
  </div>

  <div id="chat-messages">
    <div class="msg bot">Hi! I can answer questions about heart health, risk factors, and your results. What would you like to know?</div>
  </div>

  <div class="chat-input-row">
    <textarea id="chat-input" rows="1" placeholder="Ask about cholesterol, BP, risk factors…"></textarea>
    <button id="chat-send" aria-label="Send">
      <svg width="16" height="16" viewBox="0 0 24 24" fill="none">
        <path d="M22 2L11 13" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
        <path d="M22 2L15 22L11 13L2 9L22 2Z" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
      </svg>
    </button>
  </div>
</div>

<script>
  var risk = parseInt("{{ risk if risk else 0 }}") || 0;
  var bar  = document.getElementById("riskFill");
  if (bar) {
    setTimeout(function() { bar.style.width = risk + "%"; }, 300);
  }

  var toggleBtn  = document.getElementById("chat-toggle");
  var chatWindow = document.getElementById("chat-window");
  toggleBtn.addEventListener("click", function() {
    chatWindow.classList.toggle("open");
  });

  var chatInput = document.getElementById("chat-input");
  chatInput.addEventListener("input", function() {
    this.style.height = "auto";
    this.style.height = Math.min(this.scrollHeight, 100) + "px";
  });

  function sendMessage() {
    var text = chatInput.value.trim();
    if (!text) return;

    var msgs = document.getElementById("chat-messages");

    var userMsg = document.createElement("div");
    userMsg.className = "msg user";
    userMsg.textContent = text;
    msgs.appendChild(userMsg);

    chatInput.value = "";
    chatInput.style.height = "auto";

    var typing = document.createElement("div");
    typing.className = "msg typing";
    typing.id = "typing-indicator";
    typing.textContent = "Thinking…";
    msgs.appendChild(typing);
    msgs.scrollTop = msgs.scrollHeight;

    fetch("/chat", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ message: text })
    })
    .then(function(res) { return res.json(); })
    .then(function(data) {
      var t = document.getElementById("typing-indicator");
      if (t) t.remove();
      var botMsg = document.createElement("div");
      botMsg.className = "msg bot";
      botMsg.textContent = data.reply || "Sorry, I could not find an answer.";
      msgs.appendChild(botMsg);
      msgs.scrollTop = msgs.scrollHeight;
    })
    .catch(function() {
      var t = document.getElementById("typing-indicator");
      if (t) t.remove();
      var errMsg = document.createElement("div");
      errMsg.className = "msg bot";
      errMsg.textContent = "Connection error. Please try again.";
      msgs.appendChild(errMsg);
      msgs.scrollTop = msgs.scrollHeight;
    });
  }

  document.getElementById("chat-send").addEventListener("click", sendMessage);
  chatInput.addEventListener("keydown", function(e) {
    if (e.key === "Enter" && !e.shiftKey) {
      e.preventDefault();
      sendMessage();
    }
  });
</script>

</body>
</html>
"""

with open("templates/index.html", "w") as f:
    f.write(html_code)

In [20]:
import pickle
from flask import Flask, render_template, request

app = Flask(__name__)

model = pickle.load(open("health.pkl","rb"))
scaler = pickle.load(open("scaler.pkl","rb"))

feature_names = [
    'age','sex','cp','trestbps','chol','fbs',
    'restecg','thalach','exang','oldpeak',
    'slope','ca','thal'
]

@app.route('/')
def home():
    return render_template("index.html")


@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.form.to_dict()

        user_input = [float(data[col]) for col in feature_names]
        user_scaled = scaler.transform([user_input])

        prediction = model.predict(user_scaled)[0]
        prob = model.predict_proba(user_scaled)[0][1]

        result = "Heart Disease" if prediction==1 else "No Heart Disease"

        query = f"Patient has {result} with risk {round(prob*100,2)}%"
        context = retrieve(query)

        context_text = "\n".join(context)

        explanation = f"""
Prediction: {result}
Risk: {round(prob*100,2)}%

Explanation:
{context_text}

General Advice:
Maintain a healthy diet, exercise regularly, and monitor your health.
"""

        return render_template(
            "index.html",
            prediction_text=result,
            risk=round(prob*100,2),
            explanation=explanation
        )

    except Exception as e:
        print("PREDICT ERROR:", e)
        return render_template("index.html", prediction_text=str(e))

@app.route('/chat', methods=['POST'])
def chat():
    try:
        data = request.get_json()

        if not data:
            return {"reply": "No input received"}

        user_msg = data.get("message", "").strip()

        if not user_msg:
            return {"reply": "Please ask something"}

        print("User:", user_msg)

        context = retrieve(user_msg)
        context_text = "\n".join(context)

        prompt = f"""
You are a helpful medical assistant.

Use the context below to answer the question.

Context:
{context_text}

Question: {user_msg}

Explain the answer clearly in 3-4 bullet points.
Do not just repeat the context.
Give reasons and simple explanation.

Answer:
"""

        output = generator(
            prompt,
            max_new_tokens=150,
            temperature=0.7
        )

        reply = output[0]['generated_text'].strip()

        return {"reply": reply}

    except Exception as e:
        print("CHAT ERROR:", e)
        return {"reply": "Internal server error"}

with app.test_client() as client:
    test_sample = {col: 1 for col in feature_names}

    response = client.post('/predict', data=test_sample)

    if response.status_code == 200:
        print("Test Successful!")
        html_output = response.get_data(as_text=True)

        if "Heart Disease" in html_output:
            print("Result: Heart Disease")
        elif "No Heart Disease" in html_output:
            print("Result: No Heart Disease")

        import re
        risk_match = re.search(r'(\d+\.\d+)%', html_output)
        if risk_match:
            print(f"Risk: {risk_match.group(1)}%")
    else:
        print(f"Test Failed with status code: {response.status_code}")

Test Successful!
Result: Heart Disease


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [21]:
!ngrok config add-authtoken "2wQMQxsEcB1k9UhCc500EF0JVnU_2u7ivCWJs4jwJSHvDxCC9"

public_url = ngrok.connect(5000)
print("App running at:", public_url)

app.run(port=5000)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
App running at: NgrokTunnel: "https://95e3-136-111-137-138.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:56:27] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:56:28] "GET /favicon.ico HTTP/1.1" 404 -


User: causes of high bp?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:56:49] "POST /chat HTTP/1.1" 200 -


User: what can i do to lower it ?


INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:57:06] "POST /chat HTTP/1.1" 200 -


User: can you give me in some points


INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:57:28] "POST /chat HTTP/1.1" 200 -


User: thank you


INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:57:46] "POST /chat HTTP/1.1" 200 -
